# 3. Download the first ten raw videos

Download the first ten clips listed in the raw video manifest. Existing completed AVI files are skipped.

In [1]:
import csv
from pathlib import Path
from urllib.parse import quote
from urllib.request import urlopen

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data" / "raw"
VIDEO_DIRECTORY = DATA_DIR / "videos"
S3_BUCKET_URL = "https://mevadata-public-01.s3.amazonaws.com"
S3_PREFIX = "drops-123-r13"
VIDEO_LIMIT = 10
CHUNK_SIZE = 1024 * 1024

with (DATA_DIR / "video_names.csv").open(newline="", encoding="utf-8") as input_file:
    video_names = [row["video_name"] for row in csv.DictReader(input_file)][:VIDEO_LIMIT]

VIDEO_DIRECTORY.mkdir(parents=True, exist_ok=True)
downloads = []
for video_name in video_names:
    date, start_time, *_ = video_name.split(".")
    url = f"{S3_BUCKET_URL}/{quote(f'{S3_PREFIX}/{date}/{start_time[:2]}/{video_name}.r13.avi')}"
    destination = VIDEO_DIRECTORY / f"{video_name}.r13.avi"
    temporary_destination = destination.with_suffix(".avi.part")

    if destination.is_file() and destination.stat().st_size > 0:
        status = "skipped_existing"
    else:
        temporary_destination.unlink(missing_ok=True)
        with urlopen(url, timeout=60) as response, temporary_destination.open("wb") as output:
            while chunk := response.read(CHUNK_SIZE):
                output.write(chunk)
        temporary_destination.replace(destination)
        status = "downloaded"
    downloads.append({"video_name": video_name, "s3_url": url, "local_path": str(destination), "status": status})

with (DATA_DIR / "video_downloads.csv").open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=["video_name", "s3_url", "local_path", "status"])
    writer.writeheader()
    writer.writerows(downloads)

print(downloads)

[{'video_name': '2018-03-05.13-15-00.13-20-00.bus.G340', 's3_url': 'https://mevadata-public-01.s3.amazonaws.com/drops-123-r13/2018-03-05/13/2018-03-05.13-15-00.13-20-00.bus.G340.r13.avi', 'local_path': '/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/raw/videos/2018-03-05.13-15-00.13-20-00.bus.G340.r13.avi', 'status': 'skipped_existing'}, {'video_name': '2018-03-05.13-15-00.13-20-00.bus.G505', 's3_url': 'https://mevadata-public-01.s3.amazonaws.com/drops-123-r13/2018-03-05/13/2018-03-05.13-15-00.13-20-00.bus.G505.r13.avi', 'local_path': '/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/raw/videos/2018-03-05.13-15-00.13-20-00.bus.G505.r13.avi', 'status': 'skipped_existing'}, {'video_name': '2018-03-05.13-15-00.13-20-00.bus.G506', 's3_url': 'https://mevadata-public-01.s3.amazonaws.com/drops-123-r13/2018-03-05/13/2018-03-05.13-15-00.13-20-00.bus.G506.r13.avi', 'local_path': '/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/raw/vi